# Lab Linear models (simplified version)

In this version of the lab you do **not** implement the algorithms: you use the ready-made implementations of `scikit-learn`. The coding is therefore light, and the focus is on **understanding and analyzing** what the models do. Expect to be asked to justify your answers with numbers and figures.

Save the notebook as either PDF or HTML and make sure all the results are saved correctly (I won't run them and the original format does not save the results automatically), **and put your name in the filename**.

<div class="alert alert-success">
<b>Questions are in green boxes.</b>
Each one asks you to write a few lines of code (replace the <code># TODO</code>) and/or produce a figure. If you are stuck on the code for more than a few minutes, you should come see me.
</div>
<div class="alert alert-info" role="alert"><b>Analyses are in blue boxes.</b> You should comment on your results in these boxes. They are the most important part of the lab and carry most of the grade: a figure or a number without an interpretation is worth very little. Is the result good? Is it expected? Why do we get such a result? Why is it different from the previous one? Support each claim with a number or a figure.
</div>

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from math import comb
from joblib import Memory

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, Normalizer, FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import LinearSVC
from sklearn.metrics import confusion_matrix

For this lab, we will use the bluebell dataset. It consists of $64\times 64$ color images of 12 classes of flowers, which we will have to flatten into $12k$ dimensional vectors. The code for the dataset comes with several train/val/test splits: here, 5 pairs (train, val) of 1200/300 images, that we will use for cross-validation. We never touch the test set in this lab.

In [ ]:
from bluebell import Bluebell

def load_split(mode, split=0):
    ds = Bluebell('bluebell_64', mode, split=split)
    X = np.array([img.flatten()/127.5 - 1. for img, lab in ds], dtype=np.float32)
    y = np.array([lab for img, lab in ds], dtype=np.int64)
    return X, y

# the 5 (train, val) pairs, loaded once
SPLITS = [(load_split('train', k), load_split('val', k)) for k in range(5)]
(X_train, y_train), (X_val, y_val) = SPLITS[0]
print(X_train.shape, X_val.shape, np.bincount(y_train))
plt.imshow(X_train[0].reshape(64, 64, 3)/2+0.5)
print(y_train[0])

Next, we define the 0-1 loss that measures the error rate of a classifier.

In [ ]:
def error_rate(y_hat, y):
    return (1. != np.sign(y_hat*y)).mean()

def multi_error_rate(y_hat, y):
    return (1.*(np.argmax(y_hat, axis=1) != np.argmax(y, axis=1))).mean()

## The scikit-learn pattern

Every `scikit-learn` object follows the same interface:

* **estimators** (models): `model.fit(X, y)` learns the parameters, `model.predict(X)` outputs predictions. Learned parameters end with an underscore (`model.coef_`, `model.intercept_`, ...);
* **transformers** (preprocessing, e.g. `PCA`): `t.fit(X)` learns, `t.transform(X)` applies (`fit_transform` does both);
* **pipelines**: `make_pipeline(t1, t2, model)` chains them. Calling `fit` on a pipeline fits each step *on the data it is given only*, which is exactly what we want in cross-validation (the preprocessing must not see the validation data).

The toy example below fits a polynomial regression with a ridge penalty on 30 noisy points.

In [ ]:
rng = np.random.default_rng(0)
x = np.sort(rng.uniform(-1, 1, 30))[:, None]
y_toy = np.sin(3*x[:, 0]) + 0.2*rng.standard_normal(30)

model = make_pipeline(PolynomialFeatures(degree=5), Ridge(alpha=0.1))
model.fit(x, y_toy)

grid = np.linspace(-1, 1, 200)[:, None]
plt.scatter(x, y_toy, label='data')
plt.plot(grid, model.predict(grid), 'r', label='degree 5, alpha=0.1')
plt.legend()
print("coefficients of the ridge:", model[-1].coef_.round(2))

All the multi-class classifiers of this lab are built the same way: **one regressor per class** that regresses $+1$ if the image is of the class and $-1$ otherwise, and the predicted class is the `argmax` over the 12 outputs. `scikit-learn` regressors accept a matrix of 12 targets directly, so a single `fit` trains the 12 predictors.

Below is a reusable skeleton to train a model on each of the 5 splits and to cross-validate a hyper-parameter. **Read it and make sure you understand it: you will reuse it in all the questions.**

* `cross_validate(make_model, grid)` calls `make_model(value)` for each `value` of `grid` to get a (not fitted) model or pipeline, fits it on the 5 train splits, and returns the train and validation error rates, arrays of shape `(len(grid), 5)`;
* `plot_cv(grid, res)` plots the mean and standard deviation over the splits;
* `memory` caches the fitted preprocessing steps of a pipeline (use `make_pipeline(..., memory=memory)`), so a PCA is not recomputed for every value of the hyper-parameter.

In [ ]:
memory = Memory('.sk_cache', verbose=0)

def to_pm1(y, n_classes=12):
    """+1/-1 one-vs-rest targets, shape (n, n_classes)."""
    Y = -np.ones((len(y), n_classes), dtype=np.float32)
    Y[np.arange(len(y)), y] = 1.
    return Y

def scores(model, X):
    """(n, 12) class scores, whatever the type of model."""
    return model.decision_function(X) if hasattr(model, 'decision_function') else model.predict(X)

def cross_validate(make_model, grid, onehot=True, verbose=True):
    """
    For each value in `grid`, trains make_model(value) on each of the 5 train splits and
    evaluates it on the matching validation split.
    onehot=True: trains on the +1/-1 targets (regressors). onehot=False: trains on the integer labels (classifiers).
    Returns (train_err, val_err), each of shape (len(grid), 5).
    """
    train_err = np.zeros((len(grid), 5))
    val_err = np.zeros((len(grid), 5))
    for i, value in enumerate(grid):
        for k, ((Xtr, ytr), (Xva, yva)) in enumerate(SPLITS):
            model = make_model(value)
            model.fit(Xtr, to_pm1(ytr) if onehot else ytr)
            train_err[i, k] = (scores(model, Xtr).argmax(1) != ytr).mean()
            val_err[i, k] = (scores(model, Xva).argmax(1) != yva).mean()
        if verbose:
            print(f"{value}: train {train_err[i].mean():.3f} +- {train_err[i].std():.3f} | "
                  f"val {val_err[i].mean():.3f} +- {val_err[i].std():.3f}")
    return train_err, val_err

def plot_cv(grid, res, xlabel='hyper-parameter', log=True, label=''):
    train_err, val_err = res
    use_log = log and all(isinstance(g, (int, float)) and g > 0 for g in grid)
    x = np.array(grid) if use_log else np.arange(len(grid))
    for err, name in [(train_err, 'train'), (val_err, 'val')]:
        plt.errorbar(x, err.mean(1), err.std(1), capsize=3, marker='o', label=f'{label} {name}'.strip())
    if use_log:
        plt.xscale('log')
    else:
        plt.xticks(x, [str(g) for g in grid])
    plt.xlabel(xlabel); plt.ylabel('error rate'); plt.legend(); plt.grid(alpha=0.3)

Reminder: chance level for 12 balanced classes is an error rate of $11/12 \approx 0.92$.

## Linear regression on a PCA

We will first try to classify Bluebell using linear regression. Since the images have 12k dimensions (for only 1200 training images), we will first perform a PCA to reduce the dimension to something manageable.

<div class="alert alert-success"> <b> Q1.1</b> Fit a <code>sklearn.decomposition.PCA</code> with 256 components on the training set of split 0 (<code>X_train</code>), and time it. Plot the cumulative explained variance ratio (<code>explained_variance_ratio_</code>) as a function of the number of components.
</div>

In [ ]:
%%time
# TODO:

In [ ]:
# TODO: cumulative explained variance

<div class="alert alert-info" role="alert"><b>Analyze your results in this box.</b> Answer each point below (a few sentences each, backed by numbers from your results).
<ol>
<li>How many components are needed to explain 50%, 90% and 95% of the variance? Give the three numbers.</li>
<li>Compare this number to the original dimension (12288) and to the number of training images (1200). What does it tell you about the images of this dataset?</li>
<li>The PCA is <i>unsupervised</i>: it never sees the labels. Is there any guarantee that the directions of highest variance are the ones that separate the classes? Argue with what you see in Q1.2.</li>
</ol>
<b>Your answer:</b>
<br><br><br>
</div>

<div class="alert alert-success"> <b>Q1.2</b> Show the first 64 principal components (<code>pca.components_</code>) as images. Plot the reconstruction of one image per class using 1, 2, 4, 8, 16, 32 and 64 components (hint: project with <code>transform</code>, keep only the first $k$ coordinates, and multiply by the first $k$ rows of <code>components_</code>, then add <code>mean_</code>; do not forget that images are in $[-1, 1]$ and have to be rescaled to be displayed).
</div>

In [ ]:
# TODO: the 64 first components

In [ ]:
# TODO: reconstructions

<div class="alert alert-info" role="alert"><b>Analyze your results in this box.</b> Answer each point below (a few sentences each, backed by numbers from your results).
<ol>
<li>Describe what the first 3-4 components seem to encode (colors? brightness? spatial layout? shapes?). What about components 40-64?</li>
<li>From how many components is the class of the flower recognizable <i>by eye</i> in the reconstructions? Is it consistent with the variance you computed in Q1.1? Is it the same for all classes?</li>
<li>A component is a vector of 12288 numbers. Why can it be displayed as an image, and why are some pixels positive and others negative?</li>
</ol>
<b>Your answer:</b>
<br><br><br>
</div>

We now consider the bluebell dataset projected to **512 dimensions** with a PCA. We first try a simple linear regression with no regularization, with the one-vs-rest scheme described above.

<div class="alert alert-success"> <b> Q1.3</b> Use the skeleton to train a <code>LinearRegression</code> on the PCA with 512 components, on each of the 5 train splits, and evaluate it on the corresponding validation split. Then, on split 0, compute the confusion matrix on the validation set (<code>sklearn.metrics.confusion_matrix</code>) and display it.
</div>

In [ ]:
%%time
# TODO: make_model should return a pipeline PCA -> LinearRegression, use cross_validate with grid=[512]

In [ ]:
# TODO: confusion matrix on the validation set of split 0

<div class="alert alert-info" role="alert"><b>Analyze your results in this box.</b> Answer each point below (a few sentences each, backed by numbers from your results).
<ol>
<li>Report the mean and standard deviation of the train and validation error rates. How far from chance level are they? Is the standard deviation across splits small or large with respect to the differences you will observe later?</li>
<li>Compare train and validation errors. What does the gap tell you? Does the model have more parameters or more training images? How many parameters are there (512 + 1 per class)?</li>
<li>Why must the PCA be included in the pipeline (fitted on each train split only) and not fitted once on all the data before cross-validation? What would go wrong in the validation error if we did the latter?</li>
<li>Looking at the confusion matrix: which classes are well classified and which ones are confused with each other? Look at a few images of the confused classes: does the confusion make sense visually?</li>
</ol>
<b>Your answer:</b>
<br><br><br>
</div>

<div class="alert alert-success"> <b>Q1.4</b> Vary the number of PCA components in <code>[8, 32, 128, 512, 1000, 1199]</code> with an unregularized <code>LinearRegression</code>, and plot the train and validation errors as a function of the number of components.
</div>

In [ ]:
%%time
# TODO:

In [ ]:
# TODO: plot

<div class="alert alert-info" role="alert"><b>Analyze your results in this box.</b> Answer each point below (a few sentences each, backed by numbers from your results).
<ol>
<li>Describe the shape of the training and validation curves. What is the best number of components? Is the difference with the second best larger than the standard deviation across splits?</li>
<li>What happens to the training error when the number of components approaches 1199 (the number of training images minus one)? Explain it with a linear algebra argument (number of unknowns vs number of equations in the least squares problem).</li>
<li>What happens to the validation error at the same time? Explain why more features can make the classifier worse even though it can only fit the training set better.</li>
<li>What does this experiment suggest about the need for regularization? (this is the topic of the next section)</li>
</ol>
<b>Your answer:</b>
<br><br><br>
</div>

## Regularization

Next, we want to see whether regularization helps getting a better classifier. In `scikit-learn`, the strength of the regularization is the parameter `alpha` of `Ridge` and `Lasso` (the larger, the stronger). Note that `alpha=0` is not allowed in the grids below, as the problem is ill-posed.

<div class="alert alert-success"> <b> Q2.1.</b> Use <code>Ridge</code> and cross-validation to select <code>alpha</code> (a grid on a log scale, between $10^{-2}$ and $10^{5}$ is a good start; extend it if the best value is at the edge of your grid) on (a) the original bluebell dataset (12288 dimensions) and (b) the dataset reduced to 512 dimensions using a PCA in the pipeline. Plot the train and validation errors of both as a function of <code>alpha</code>.
</div>

In [ ]:
%%time
alpha_ridge_grid = [1e-2, 1e-1, 1., 1e1, 1e2, 1e3, 1e4, 1e5]
# TODO: raw pixels

In [ ]:
%%time
# TODO: PCA 512

In [ ]:
# TODO: plot both

<div class="alert alert-info" role="alert"><b>Analyze your results on the full training set in this box.</b> Answer each point below (a few sentences each, backed by numbers from your results).
<ol>
<li>Report for each of the two settings the best <code>alpha</code> and the corresponding validation error (mean +- std). Is the best <code>alpha</code> in the interior of your grid? (If not, extend it.)</li>
<li>Describe the curves: what happens to the training error and the validation error when alpha is very small? when it is very large? Explain why in terms of what the penalty does to the weights.</li>
<li>Compare raw pixels and PCA-512. Which one is better? Is the difference significant with respect to the standard deviation across splits? Which one is faster and why? Why does ridge on the full 12288-dimensional data not suffer from the problem seen in Q1.4 when the number of features is larger than the number of images?</li>
<li>Does regularization help with respect to the unregularized regression of Q1.3? By how much?</li>
</ol>
<b>Your answer:</b>
<br><br><br>
</div>

<div class="alert alert-success"> <b>Q2.2.</b> For the best <code>alpha</code> on the raw pixels, fit a <code>Ridge</code> on split 0 and display the weight vectors of 3 classes of your choice as images (<code>coef_[c]</code> has 12288 values; reshape to $64\times64\times3$ and rescale for display, or sum over the color channels). Also plot the norm of the weights <code>np.linalg.norm(coef_)</code> as a function of <code>alpha</code> (log scale).
</div>

In [ ]:
# TODO: weight images

In [ ]:
# TODO: weight norm vs alpha

<div class="alert alert-info" role="alert"><b>Analyze your results in this box.</b> Answer each point below (a few sentences each, backed by numbers from your results).
<ol>
<li>What do the weight images of the three classes look like? Can you recognize something of the class (a color, a shape, a typical background)? What does this tell you about what the linear classifier relies on?</li>
<li>How does the norm of the weights vary with alpha? Relate it to the validation curve of Q2.1.</li>
<li>Ridge is known to <i>shrink</i> the weights along the directions of low variance of the data more than along those of high variance. Using this, explain why ridge on the raw pixels and ridge on PCA-512 end up with similar performance.</li>
</ol>
<b>Your answer:</b>
<br><br><br>
</div>

Similarly, we now want to try the LASSO, which penalizes the $\ell_1$ norm of the weights instead of the squared $\ell_2$ norm. Be careful: the LASSO has no closed-form solution and is slower to train (`scikit-learn` uses coordinate descent), especially on 12288 dimensions and for small `alpha` (a single fit with `alpha=1e-4` on the raw pixels takes over a minute!). `ConvergenceWarning`s are expected: you can ignore them in this lab. This is why the grid for the raw pixels below is restricted to larger values of `alpha` than the one you can afford in the PCA space.

<div class="alert alert-success"> <b> Q2.3.</b> Use <code>Lasso</code> and cross-validation to select <code>alpha</code> on (a) the original bluebell dataset and (b) the dataset reduced to 512 dimensions using a PCA in the pipeline. Plot the train and validation errors of both as a function of <code>alpha</code>. For both, also report the fraction of weights that are exactly zero (<code>coef_ == 0</code>) for each <code>alpha</code> on split 0.
</div>

In [ ]:
%%time
alpha_lasso_grid_raw = [5e-3, 1e-2, 2e-2, 5e-2]
# TODO: raw pixels

In [ ]:
%%time
alpha_lasso_grid_pca = [1e-4, 1e-3, 1e-2, 3e-2, 1e-1]
# TODO: PCA 512

In [ ]:
# TODO: plots, and fraction of zero weights vs alpha

<div class="alert alert-info" role="alert"><b>Analyze your results on the full training set in this box.</b> Answer each point below (a few sentences each, backed by numbers from your results).
<ol>
<li>Report the best <code>alpha</code> and the corresponding validation error for both settings. Compare with ridge (Q2.1): which is better, and is the difference larger than the standard deviation across splits?</li>
<li>Plot or tabulate the fraction of zero weights as a function of <code>alpha</code>. How does the validation error behave as sparsity increases? What is the best sparsity level, and how many pixels (or components) does the best model still use?</li>
<li>What happens when alpha is too large (look at the predictions of the model)? Why?</li>
<li>Why do you think the LASSO is relatively better or worse in the PCA space than in the pixel space? (Think about what a zero weight means in each space.)</li>
</ol>
<b>Your answer:</b>
<br><br><br>
</div>

<div class="alert alert-success"> <b>Q2.4.</b> For the best <code>alpha</code> on the raw pixels, display as images the (absolute value of the) weights of the same 3 classes as in Q2.2, and compare with the ridge weights.
</div>

In [ ]:
# TODO:

<div class="alert alert-info" role="alert"><b>Analyze your results in this box.</b> Answer each point below (a few sentences each, backed by numbers from your results).
<ol>
<li>Describe the differences between the LASSO and the ridge weight images. Where are the non-zero LASSO weights located, and does it make sense given the images of the class?</li>
<li>In which situations would you prefer the LASSO to ridge? Is this one of them?</li>
</ol>
<b>Your answer:</b>
<br><br><br>
</div>

## Generalized linear models

A linear classifier is unlikely to be able to classify correctly all classes, we will thus try a generalized linear model using a non-linear projection. We use for each input dimension the polynomial features $\phi(x) = [x, x^2, \dots, x^{d}]$ concatenated over the input dimensions. The helper below does it (it is applied after `Normalizer`, that gives each vector a unit $\ell_2$ norm).

<div class="alert alert-success"> <b> Q3.1.</b> <code>sklearn.preprocessing.PolynomialFeatures</code> computes <i>all</i> the monomials of degree up to $d$, including the products between different dimensions. Compute (with the formula or with <code>math.comb</code>) how many features it would create from 512 dimensions for degrees 1, 2, 3 and 5, and compare with the number of features created by <code>per_dim_poly</code>. Check on a small example (e.g. 5 dimensions) that your formulas match the output of the two transformers.
</div>

In [ ]:
def _powers(X, degree):
    return np.hstack([X**k for k in range(1, degree+1)])

def per_dim_poly(degree):
    """phi(x) = [x, x^2, ..., x^degree], concatenated over the input dims."""
    return FunctionTransformer(_powers, kw_args={'degree': degree})

In [ ]:
# TODO:

<div class="alert alert-info" role="alert"><b>Analyze your results in this box.</b> Answer each point below (a few sentences each, backed by numbers from your results).
<ol>
<li>Give the table of number of features for the two ways of building polynomial features, for degrees 1, 2, 3, 5.</li>
<li>Compare the number of features with the number of training images (1200). What does this imply for the unregularized regression? Why is the full <code>PolynomialFeatures</code> unusable in practice here, and what do we lose by using <code>per_dim_poly</code> instead?</li>
</ol>
<b>Your answer:</b>
<br><br><br>
</div>

We consider only the **512 dimension** PCA of the dataset for the following questions, and we normalize each vector **to have unit l2 norm** with `Normalizer()` (after the PCA, before the polynomial features).

<div class="alert alert-success"> <b>Q3.2.</b> Using cross validation, find the best degree to classify bluebell using unregularized <code>LinearRegression</code> (pipeline: PCA 512 -> Normalizer -> per_dim_poly -> LinearRegression), for the degrees in <code>[1, 2, 3, 4, 6, 8]</code>.
</div>

In [ ]:
%%time
degree_grid = [1, 2, 3, 4, 6, 8]
# TODO:

In [ ]:
# TODO: plot

<div class="alert alert-info" role="alert"><b>Analyze your results on the full training set in this box.</b> Answer each point below (a few sentences each, backed by numbers from your results).
<ol>
<li>Which degree is best? Report the validation error and compare to the linear model (degree 1). Is the gain larger than the standard deviation across splits?</li>
<li>How many features are there for each degree? At which degree do you have more features than training images? What happens to the training error there, and to the validation error?</li>
<li>Is the non-linear model helping at all here? Give a reason why a polynomial of the PCA coordinates may or may not be a good model for flower images.</li>
</ol>
<b>Your answer:</b>
<br><br><br>
</div>

<div class="alert alert-success"> <b>Q3.3</b> Using a maximum degree of 5 and <code>Ridge</code>, perform cross-validation on <code>alpha</code> and classify bluebell (same pipeline as Q3.2 with <code>Ridge</code> instead of <code>LinearRegression</code>). Note that the scale of the features changes a lot with the degree: the grid below is a starting point, extend it if necessary.
</div>

In [ ]:
%%time
alpha_ridge_grid_poly = [1e-4, 1e-3, 1e-2, 1e-1, 1., 10.]
# TODO:

In [ ]:
# TODO: plot

<div class="alert alert-info" role="alert"><b>Analyze your results on the full training set in this box.</b> Answer each point below (a few sentences each, backed by numbers from your results).
<ol>
<li>Report the best <code>alpha</code> and its validation error. Compare with (i) unregularized degree 5 of Q3.2, (ii) the best linear ridge of Q2.1(b).</li>
<li>The best <code>alpha</code> is not the same as in Q2.1(b). Why? (Hint: look at the magnitude of the features after the normalization and the powers: print the standard deviation of the features of degree 1 and of degree 5.)</li>
<li>Does the combination polynomial features + regularization beat each of them alone? Explain.</li>
</ol>
<b>Your answer:</b>
<br><br><br>
</div>

<div class="alert alert-success"> <b>Q3.4</b> Using a maximum degree of 5 and <code>Lasso</code>, perform cross-validation on <code>alpha</code> and classify bluebell. Report the fraction of zero weights, and the fraction of zero weights per degree (1 to 5) for the best <code>alpha</code> on split 0 (hint: the features are ordered by degree, in blocks of 512).
</div>

In [ ]:
%%time
alpha_lasso_grid_poly = [1e-5, 1e-4, 1e-3, 1e-2]
# TODO:

In [ ]:
# TODO: plot, and sparsity per degree

<div class="alert alert-info" role="alert"><b>Analyze your results on the full training set in this box.</b> Answer each point below (a few sentences each, backed by numbers from your results).
<ol>
<li>Report the best <code>alpha</code> and its validation error. Compare with ridge in Q3.3 and with the linear LASSO of Q2.3(b).</li>
<li>Which degrees does the LASSO keep? Is it consistent with your answers to Q3.2 and Q3.3? What does it say about the usefulness of high-degree features?</li>
<li>Summarize in a table the best validation error of every model of this lab (linear regression, ridge, LASSO, and the polynomial versions). Which model would you pick to deploy and why? What additional experiment would you run before trusting this choice (hint: what have we not used yet)?</li>
</ol>
<b>Your answer:</b>
<br><br><br>
</div>

## Bonus

The questions in this section are bonus, they don't bring any additional point.

<div class="alert alert-warning"> <b>Q4.1 [Optional].</b> Use <code>sklearn.linear_model.ElasticNet</code>, that combines ridge and LASSO (parameters <code>alpha</code> and <code>l1_ratio</code>), to select both hyper-parameters by cross-validation on the polynomial features of degree 5.</div>

In [ ]:
%%time
# TODO:

<div class="alert alert-info" role="alert"><b>Analyze your results in this box.</b> Answer each point below (a few sentences each, backed by numbers from your results).
<ol>
<li>Which combination is best? Is the elastic net better than both ridge and LASSO alone (Q3.3, Q3.4), by more than the standard deviation across splits? Which behavior (sparsity, shrinkage) does the best combination keep?</li>
</ol>
<b>Your answer:</b>
<br><br><br>
</div>

<div class="alert alert-warning"> <b>Q4.2 [Optional].</b> Use <code>sklearn.svm.LinearSVC</code> (linear SVM, hinge loss, one-vs-rest) on the polynomial features of degree 5, and use cross-validation to set its regularization parameter <code>C</code> (note that <code>C</code> is the <i>inverse</i> of the regularization strength). Remember that <code>LinearSVC</code> is trained on the integer labels: use <code>cross_validate(..., onehot=False)</code>.</div>

In [ ]:
%%time
C_grid = [1e-3, 1e-2, 1e-1, 1., 10.]
# TODO:

<div class="alert alert-info" role="alert"><b>Analyze your results in this box.</b> Answer each point below (a few sentences each, backed by numbers from your results).
<ol>
<li>Compare with ridge on the same features (Q3.3). Both are linear models with an $\ell_2$ penalty: what is the difference between them, and does it matter here? How does the best <code>C</code> relate to the best <code>alpha</code> of ridge?</li>
</ol>
<b>Your answer:</b>
<br><br><br>
</div>